In [ ]:
!pip install py-solc-x

In [ ]:
!pip install solc-select

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 29.6 MB/s eta 0:00:00


In [ ]:
import re

def extract_solidity_version(file_path):
    try:
        with open(file_path, 'r') as file:
            content = file.read()

        # Look for the first version number in the pragma line
        match = re.search(r'pragma\s+solidity\s+[^;]*?(\d+\.\d+\.\d+)', content)
        if match:
            version = match.group(1)
            print(f"{version}")
            return version
        else:
            print("No Solidity version found.")
            return None
    except Exception as e:
        print(f"Error: {e}")
        return None


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving contract_20260103_223109.sol to contract_20260103_223109 (2).sol


In [ ]:
file_path = list(uploaded.keys())[0]
ver = extract_solidity_version(file_path)

0.4.24


In [ ]:
!solc-select install {ver}

Installing solc '0.4.24'...
Version '0.4.24' installed.


In [ ]:
!solc-select use {ver}

Switched global version to 0.4.24


In [ ]:
#Install Solidity Compiler
from solcx import install_solc
install_solc("0.4.24")

<Version('0.4.24')>

In [ ]:
import json
from solcx import compile_standard

def get_ast(solidity_file):

    with open(solidity_file, "r") as f:
        source = f.read()

    compiled = compile_standard(
        {
            "language": "Solidity",
            "sources": {
                "contract_20260103_223109 (2).sol": {"content": source}
            },
            "settings": {
                "outputSelection": {
                    "*": {
                        "": ["ast"]
                    }
                }
            },
        },
        solc_version= ver,
    )

    return compiled["sources"]["contract_20260103_223109 (2).sol"]["ast"]


def extract_modifiers(ast):

    modifier_definitions = []
    modifier_usage = []

    def traverse(node):

        if isinstance(node, dict):

            if node.get("nodeType") == "ModifierDefinition":
                modifier_definitions.append(node["name"])

            if node.get("nodeType") == "FunctionDefinition":
                modifiers = node.get("modifiers", [])

                for m in modifiers:
                    modifier_usage.append({
                        "function": node.get("name"),
                        "modifier": m["modifierName"]["name"]
                    })

            for key in node:
                traverse(node[key])

        elif isinstance(node, list):
            for item in node:
                traverse(item)

    traverse(ast)

    return modifier_definitions, modifier_usage


file = "/content/contract_20260103_223109 (2).sol"   # uploaded file name

ast = get_ast(file)

defs, usage = extract_modifiers(ast)

print("Modifier Definitions:")
for m in defs:
    print("-", m)

print("\nModifier Usage:")
for u in usage:
    print(f"Function {u['function']} uses modifier {u['modifier']}")

Modifier Definitions:
- onlyOwner
- onlyManager
- onlyPendingOwner
- onlyWhitelisted
- whenNotPaused
- whenPaused
- canMint
- whenNotLocked

Modifier Usage:
Function transferOwnership uses modifier onlyOwner
Function claimOwnership uses modifier onlyPendingOwner
Function setManager uses modifier onlyOwner
Function addAddressToWhitelist uses modifier onlyOwner
Function addAddressesToWhitelist uses modifier onlyOwner
Function removeAddressFromWhitelist uses modifier onlyOwner
Function removeAddressesFromWhitelist uses modifier onlyOwner
Function pause uses modifier onlyOwner
Function pause uses modifier whenNotPaused
Function unpause uses modifier onlyOwner
Function unpause uses modifier whenPaused
Function transfer uses modifier whenNotPaused
Function transferFrom uses modifier whenNotPaused
Function approve uses modifier whenNotPaused
Function increaseApproval uses modifier whenNotPaused
Function decreaseApproval uses modifier whenNotPaused
Function mint uses modifier onlyManager
Funct